3.5 調整切段處理

In [1]:
import whisper
import paddlehub as hub
import os
import csv
import time
import logging
import gc
import torch
from datetime import datetime
from pydub import AudioSegment
from opencc import OpenCC
from docx import Document
from pathlib import Path
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

whisper_model = whisper.load_model("large")
punc_model = hub.Module(name='auto_punc')

def convert_to_wav(input_file: str):
    base = Path(input_file).stem
    output_file = Path(input_file).with_name(f"{base}.wav")
    audio = AudioSegment.from_file(input_file)
    wav_audio = audio.set_channels(1).set_frame_rate(44100)
    wav_audio.export(output_file, format="wav")
    logging.info(f"轉換 WAV 成功: {output_file}")
    return str(output_file)

def my_whisper(audio_path, segment_length=300000):
    logging.info("開始進行中文語音辨識（分段+進度提示）")
    audio = AudioSegment.from_wav(audio_path)
    segments = [audio[i:i+segment_length] for i in range(0, len(audio), segment_length)]

    # 在此加入分段數量提示
    total_segments = len(segments)
    logging.info(f"音檔已分割成 {total_segments} 個分段進行處理。")

    full_text = ""
    for idx, segment in enumerate(tqdm(segments, desc="辨識進度")):
        segment.export("temp.wav", format="wav")
        result = whisper_model.transcribe("temp.wav", language='zh')
        full_text += result["text"]

        os.remove("temp.wav")
        del segment, result
        torch.cuda.empty_cache()
        gc.collect()

    logging.info("中文語音辨識完成")
    return full_text

def check_audio_file(file_name, input_file_path, is_word_correcting="N", correct_csv="ReplaceWord_v1.0.csv"):
    supported_extensions = ['aac', 'wav', 'mp3', 'm4a', 'mp4']
    for ext in supported_extensions:
        potential_file = Path(input_file_path) / f"{file_name}.{ext}"
        if potential_file.is_file():
            logging.info(f"音檔確認成功: {potential_file}")
            return str(potential_file), is_word_correcting.upper(), correct_csv
    raise FileNotFoundError("找不到音檔")

def ch_convert(transcript, method):
    return OpenCC(method).convert(transcript)

def add_punctuation(raw_script):
    def split_text(text, max_length=200):
        return [text[i:i+max_length] for i in range(0, len(text), max_length)]
    text_list = split_text(raw_script)
    processed_text = punc_model.add_puncs(text_list)
    logging.info('標點符號加入完成')
    return "".join(processed_text)

def fix_wording(fix_txt, csv_file):
    with open(csv_file, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        replace_dict = {rows[0]: rows[1] for rows in reader}
    for old_word, new_word in replace_dict.items():
        fix_txt = fix_txt.replace(old_word, new_word)
    logging.info('文字更正置換完成')
    return fix_txt.replace("-", "\n")

def convert_text(input_text, file_name, output_file_path):
    output_text = Path(output_file_path) / f"{file_name}.txt"
    with open(output_text, 'w', encoding='utf-8') as file:
        file.write(input_text)
    logging.info(f"已輸出純文字檔: {output_text}")

def convert_word(input_text, file_name, output_file_path):
    output_word = Path(output_file_path) / f"{file_name}.docx"
    doc = Document()
    doc.add_paragraph(input_text)
    doc.save(output_word)
    logging.info(f"已輸出 Word 檔: {output_word}")

def main(file_list, input_file_path, is_word_correcting, correct_csv):
    for file_name in file_list:
        logging.info(f"🚩 開始處理檔案：{file_name}")
        start_time = time.time()

        input_file, is_correcting, correct_csv = check_audio_file(
            file_name, input_file_path, is_word_correcting, correct_csv
        )

        # 強制轉換成 WAV
        wav_path = convert_to_wav(input_file)
        transcript = my_whisper(wav_path)

        simplified_text = ch_convert(transcript, 'tw2s')
        processed_transcript = ch_convert(add_punctuation(simplified_text), 's2tw')

        convert_text(transcript, file_name + '_Raw', input_file_path)

        final_result = (fix_wording(processed_transcript, correct_csv)
                        if is_correcting == "Y" else processed_transcript)

        convert_text(final_result, file_name, input_file_path)
        convert_word(final_result, file_name, input_file_path)

        # 刪除中間 WAV 檔
        if os.path.exists(wav_path):
            os.remove(wav_path)
            logging.info(f"已刪除中間WAV檔案：{wav_path}")

        execution_time = (time.time() - start_time) / 60
        logging.info(f"✅ 檔案 {file_name} 處理完成，耗時：{execution_time:.2f} 分鐘\n")

    logging.info("🎉 所有檔案處理完畢！🎉")


C:\Users\sunri\anaconda3\envs\py38\lib\site-packages\whisper\timing.py:58: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def backtrace(trace: np.ndarray):
C:\Users\sunri\anaconda3\envs\py38\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
[2025-04-13 12:24:25,757] [    INFO] - loading configuration file C:\Users\sunri\.paddlehub\modules\auto_punc\assets\ckpt\config.json
[2025-04-13 12:24:25,759] [    INFO] - Model config ErnieConfig {
  "architectures": [
    "ErnieForTokenClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "enable_recompute": false,
  "fuse": fa

In [2]:


# 傳入多個檔名
file_list = [
    "畢大_20250412_紹興"
]


for file_name in file_list:
    main(
        file_list=[file_name],
        input_file_path="C:/Users/sunri/Desktop/採訪/",
        is_word_correcting="N",
        correct_csv="ReplaceWord_v1.0.csv"
    )

    # 處理完成後清除 PyTorch 記憶體
    torch.cuda.empty_cache()
    gc.collect()


2025-04-13 12:24:34,402 - INFO - 🚩 開始處理檔案：畢大_20250412_紹興
2025-04-13 12:24:34,405 - INFO - 音檔確認成功: C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興.m4a
2025-04-13 12:25:09,855 - INFO - 轉換 WAV 成功: C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興.wav
2025-04-13 12:25:09,941 - INFO - 開始進行中文語音辨識（分段+進度提示）
2025-04-13 12:25:10,556 - INFO - 音檔已分割成 23 個分段進行處理。
辨識進度: 100%|█████████████████████████████████████████████████████████████████████| 23/23 [2:27:01<00:00, 383.54s/it]
2025-04-13 14:52:11,907 - INFO - 中文語音辨識完成
2025-04-13 14:52:14,671 - INFO - 標點符號加入完成
2025-04-13 14:52:14,931 - INFO - 已輸出純文字檔: C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興_Raw.txt
2025-04-13 14:52:14,931 - INFO - 已輸出純文字檔: C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興.txt
2025-04-13 14:52:14,993 - INFO - 已輸出 Word 檔: C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興.docx
2025-04-13 14:52:15,067 - INFO - 已刪除中間WAV檔案：C:\Users\sunri\Desktop\採訪\畢大_20250412_紹興.wav
2025-04-13 14:52:15,071 - INFO - ✅ 檔案 畢大_20250412_紹興 處理完成，耗時：147.68 分鐘

2025-04-13 14:52:15,071 - INFO - 

In [37]:
# 字太多，在Jupter看時，用分段看
def print_seg(text, word_len):
    for i in range(0, len(text), word_len):
        print(text[i:i+word_len] +'\n')
        print( '=' * 5 + str(i) +  '=' * 100 +'\n')

print_seg(final_result, 500)

對，然後，在做子宮切除的時候，因為有一個叫做uterineartery，然後會經過子宮，應該是一個動脈，需要剪斷，但是它離輸尿管還蠻近的，所以在子宮切除的過程中，可能會誤傷到輸尿，泌尿系統的相關構造，所以我們就想說，那這是不是可以去解決的？然後，畢竟現在ai那麼當紅，然後，我們可不可以，就是有點類似其他人的問題，我們可不可以解決這個問題？這樣子，然後，主要問題是這樣，不要講這樣的，所以，簡單來講，你們想發想，這個原因，就是因為達文西手臂現在的狀況是。大部分是以切子宮手術為主，但是切的過程當中，很容易去傷到泌尿，輸尿管，是嗎？泌尿系統：對，所以想要解，應該會出現，大宗，應該不算是子宮切除，就是因為他現在大部分施術系統，都只是就是給你一套可以動手術的系統，他不會有一個警示，說，你可能切到了血管，你可能切到了輸尿管，你可能切到你不該切到的東西的時候，他不會有任何的feed、back，給你證實，讓你醫生可以知道，那我們現在就想要做一個像是額外的、附屬的系統，就可以讓他鑲嵌在我們原本打印手術上的打印。手臂上的一個輔助系統，讓他可以有更多的功能性，然後可能希望之後會是可以整合到整個打印手臂，整個生

=====0====================================================================================================

態系裡面，而不是隻有兩個外掛程式，一樣，目標是想要講的，喔！好，這樣，非常清楚明瞭，就是因為我的確有一個朋友，他的確也是拆除子宮的時候，也是用達文西，然後他先生有進去看，他說，現在技術真的超厲害的，就是根本就是還可以補一個網進去，就是，我也不知道是什麼網，對，反正他就很驚歎，那個，只是現階段的達文西手術，達文西手臂，他沒有辦法告訴你，說，你可能會誤傷哪。裡，但是，你們就是想解決這個問題，把它開發一個附屬系統，一個輔助達文西手臂的東西，是嗎？對喔！好，因為我們還有做一些！對，然後呢，就是，我們有做一些文獻回顧，就是看到很多那個數字，要講美國的數字，你還記得數字嗎？你可以稍微提一下，我忘記這個數字是多少，你是說受傷的數嗎？對，就是48，喔！就是在我們做實工切除的時候，有48的病患，會被那個不小心切到，不該切到的地方，像是塑料管，或是血管，然後，我們想說，可以把它降低一下，然

In [38]:
## 錯字再調整區
#確認是否要進入改錯字流程
if is_word_correcting == "Y":
    final_result = fix_wording(processed_transcript,correct_csv)
else:
    final_result = processed_transcript
    

# 轉入純文字檔 & word檔
convert_text(final_result,file_name,input_file_path)
convert_word(final_result,file_name,input_file_path)

print_seg(final_result, 500)

文字更正置換完成

文本已输出到 C:/Users/sunri/OneDrive/桌面/採訪/voice_1254968.txt 純文字檔中
文本已输出到 C:/Users/sunri/OneDrive/桌面/採訪/voice_1254968.docx 文件中。
對，然後，在做子宮切除的時候，因為有一個叫做uterineartery，然後會經過子宮，應該是一個動脈，需要剪斷，但是它離輸尿管還蠻近的，所以在子宮切除的過程中，可能會誤傷到輸尿，泌尿系統的相關構造，所以我們就想說，那這是不是可以去解決的？然後，畢竟現在ai那麼當紅，然後，我們可不可以，就是有點類似其他人的問題，我們可不可以解決這個問題？這樣子，然後，主要問題是這樣，不要講這樣的，所以，簡單來講，你們想發想，這個原因，就是因為達文西手臂現在的狀況是。大部分是以切子宮手術為主，但是切的過程當中，很容易去傷到泌尿，輸尿管，是嗎？泌尿系統：對，所以想要解，應該會出現，大宗，應該不算是子宮切除，就是因為他現在大部分施術系統，都只是就是給你一套可以動手術的系統，他不會有一個警示，說，你可能切到了血管，你可能切到了輸尿管，你可能切到你不該切到的東西的時候，他不會有任何的feed、back，給你證實，讓你醫生可以知道，那我們現在就想要做一個像是額外的、附屬的系統，就可以讓他鑲嵌在我們原本打印手術上的打印。手臂上的一個輔助系統，讓他可以有更多的功能性，然後可能希望之後會是可以整合到整個打印手臂，整個生

=====0====================================================================================================

態系裡面，而不是隻有兩個外掛程式，一樣，目標是想要講的，喔！好，這樣，非常清楚明瞭，就是因為我的確有一個朋友，他的確也是拆除子宮的時候，也是用達文西，然後他先生有進去看，他說，現在技術真的超厲害的，就是根本就是還可以補一個網進去，就是，我也不知道是什麼網，對，反正他就很驚歎，那個，只是現階段的達文西手術，達文西手臂，他沒有辦法告訴你，說，你可能會誤傷哪。裡，但是，你們就是想解決這個問題，把它開發一個附屬系統，一個輔助達文西手臂的東西，是嗎？對喔！好，因為我們還有做一些！對，然後呢，就是，我們有做一些文獻回顧，就